# Stage 2 analysis (B4)

RQ2 (computed vs fabricated), RQ3 (layers and heads) and the bridge to Stage 1 (lookback by decoding setting). Run after B3. The numbers come from `analysis/stage2.py` (definitions at the top of that file); what they mean is ours to write (README §3).

In [ ]:
import sys
from pathlib import Path

# Find the repo root, whether the notebook runs from the root or from analysis/.
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("run this notebook inside the repo")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, Markdown, display

from common.io import load_config, run_dir
from analysis.stage2 import run_all

# Another model: point CONFIG at its file, e.g. ROOT / "configs" / "llama1b.yaml" (MANUAL_STEPS.md,
# "Other models"); the folders below then become gen/<run_name>/, labels/<run_name>/, ...
CONFIG = ROOT / "config.yaml"
cfg = load_config(CONFIG)
DIR = {f: run_dir(cfg, ROOT, f) for f in ["gen", "labels", "baseline", "features", "detector", "analysis/figures"]}

# Where the B2 feature files are: the repo's features/ folder, or our shared Drive folder
# (MANUAL_STEPS.md), e.g. Path("/content/drive/MyDrive/GoG-shared/features").
FEATURES_DIR = DIR["features"]

In [ ]:
tables = run_all(cfg, ROOT / "data" / "finqa_subset.jsonl", FEATURES_DIR, DIR["baseline"],
                 DIR["detector"], DIR["analysis/figures"])

## RQ2: do the detectors confuse computed numbers with fabricated ones?

AUROC of fabricated vs copied and fabricated vs computed, separately, and the share of each label's test numbers each detector flags at its val-tuned threshold (computed = correct computed numbers).

In [ ]:
display(Markdown((DIR["analysis/figures"] / "stage2_rq2.md").read_text()))
pd.DataFrame(tables["rq2"]).set_index("detector").round(3)

### Our reading (RQ2)

_In our own words._

## RQ3: which layers and heads carry the signal?

In [ ]:
display(Image(str(DIR["analysis/figures"] / "stage2_layers.png")))
display(pd.DataFrame(tables["topk"]).round(3))
display(Image(str(DIR["analysis/figures"] / "stage2_heads.png")))
pd.DataFrame(tables["heads"]).groupby("feature_set").head(10)

### Our reading (RQ3)

_In our own words, including: does a top-100-head detector keep most of the AUROC?_

## Bridge: lookback ratio by decoding setting

In [ ]:
display(Image(str(DIR["analysis/figures"] / "stage2_bridge.png")))
pd.DataFrame(tables["bridge"]).pivot(index="setting", columns="label", values="mean_lookback").round(3)

### Our reading (bridge)

_In our own words: does the lookback ratio fall as temperature rises?_